# EIM-Agent 云端 GPU 补充实验（Colab / 魔搭 / AutoDL 通用）

用途：在 GPU 上用更大的生成模型（Qwen3-14B / 32B）复现第 7 章问答实验，作为“不同规模生成模型”的对比。

**数据安全：仅上传 SimEIM 虚构数据集与已构建的索引（`experiments/runs/index/full`），不得上传任何企业真实数据。**

步骤：1) 上传项目压缩包 `eim-agent.zip`（本地执行 `zip -r eim-agent.zip eim-agent -x '*/.venv/*' '*/node_modules/*'`）；2) 依次运行下列单元。

In [ ]:
!nvidia-smi
!unzip -q -o eim-agent.zip && cd eim-agent && pip -q install -e . && pip -q install vllm

In [ ]:
# 启动 vLLM 的 OpenAI 兼容服务（T4 16GB 建议 Qwen3-8B；A100 可用 Qwen3-14B/32B-AWQ）
import subprocess, time
MODEL = 'Qwen/Qwen3-14B-AWQ'
srv = subprocess.Popen(['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', MODEL,
                        '--served-model-name', 'qwen3-gpu', '--max-model-len', '16384', '--port', '8001'])
time.sleep(240)

In [ ]:
import os
os.environ.update({'EIM_LLM_BASE_URL': 'http://127.0.0.1:8001/v1', 'EIM_LLM_API_KEY': 'x', 'EIM_LLM_MODEL': 'qwen3-gpu',
                   'EIM_DEVICE': 'cuda', 'EIM_RUNS': '/content/eim-agent/experiments/runs_gpu'})
!mkdir -p /content/eim-agent/experiments/runs_gpu/index && cp -r /content/eim-agent/experiments/runs/index/full /content/eim-agent/experiments/runs_gpu/index/
# 评审模型保持与本地实验一致（如需更公平，可另起一个服务作为评审）
!cd eim-agent/experiments && python eval_qa.py --variant full --systems 'NaiveRAG,Ours(full)'

In [ ]:
# 下载结果文件，拷回本地 experiments/runs_gpu/ 后在论文中引用
!cd eim-agent/experiments && zip -r runs_gpu.zip runs_gpu/qa_full.json